In [5]:
# ============================================================
# CELL 1: Environment Setup & Repository Cloning
# ============================================================

# 1. Change working directory to Kaggle's writable storage
import os
os.chdir('/kaggle/working')

# 2. Remove any previous clone (optional: prevents conflicts from partial clones)
#    'rm -rf' works on Kaggle because it's a Linux environment.
!rm -rf /kaggle/working/Emergence-of-Thinking

# 3. Clone the official paper repository
!git clone https://github.com/GuanghaoYe/Emergence-of-Thinking.git

# 4. Move into the cloned repository
os.chdir('/kaggle/working/Emergence-of-Thinking')

# 5. Fix Ray version from 2.12.0 → 2.31.0 in all config files
#    (Ray 2.12 has issues; 2.31 is compatible with Kaggle's Python)
print('---> Fixing Ray version references...')
for root, dirs, files in os.walk('.'):
    if '.git' in root:
        continue
    for file in files:
        if file.endswith(('.txt', '.py', '.toml')):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, 'r') as f:
                    content = f.read()
                if 'ray' in content.lower() and '2.12.0' in content:
                    with open(filepath, 'w') as f:
                        f.write(content.replace('2.12.0', '2.31.0'))
                    print(f'  Fixed: {filepath}')
            except:
                pass

# ============================================================
# CELL 2: Install Dependencies (Kaggle-optimized, with fixes)
# ============================================================

# 1. Upgrade pip, setuptools, wheel
print('\n---> Upgrading pip and build tools...')
!pip install --upgrade pip setuptools wheel --user -q

# 2. Install PyTorch (already present, but ensure compatibility)
print('\n---> Installing PyTorch...')
!pip install torch torchvision torchaudio --user -q

# 3. Install build dependencies
print('\n---> Installing common build dependencies...')
!pip install numpy cython packaging --user -q

# 4. Remove flash-attn from requirements.txt to avoid build failure
print('\n---> Removing flash-attn from requirements.txt...')
req_file = '/kaggle/working/Emergence-of-Thinking/requirements.txt'
if os.path.exists(req_file):
    with open(req_file, 'r') as f:
        lines = f.readlines()
    with open(req_file, 'w') as f:
        for line in lines:
            if 'flash-attn' not in line.lower():
                f.write(line)
    print('  Removed flash-attn from requirements.txt')

# 5. Skip flash-attn build
os.environ['FLASH_ATTENTION_SKIP_CUDA_BUILD'] = 'TRUE'

# 6. Try installing the core project (without flash-attn dependency)
print('\n---> Attempting editable install...')
!pip install -e . --user -q

# 7. If editable fails, try regular install
if os.system('pip show emergence-of-thinking > /dev/null 2>&1') != 0:
    print('\n---> Editable install failed. Trying regular install...')
    !pip install . --user -q

# 8. Add repo to Python path
import sys
sys.path.insert(0, '/kaggle/working/Emergence-of-Thinking')
print("\n✅ Setup complete. Repository added to Python path.")

Cloning into 'Emergence-of-Thinking'...
remote: Enumerating objects: 7063, done.
remote: Total 7063 (delta 0), reused 0 (delta 0), pack-reused 7063 (from 1)
Receiving objects: 100% (7063/7063), 11.16 MiB | 18.96 MiB/s, done.
Resolving deltas: 100% (5040/5040), done.
---> Fixing Ray version references...
  Fixed: ./requirements.txt

---> Upgrading pip and build tools...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.9 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you h

In [7]:
!pip install transformers==4.46.3 accelerate --user -q


In [14]:
import os
import shutil

# 1. Your exact dataset path from the sidebar
# UPLOAD->DATASET->length_log.csv&length.pkl->ONOMATIZW DATASET->CP DATASET_DIR
input_dataset_dir = "/kaggle/input/datasets/livanasdimitris/1024-70steps/"  # TODO: NA TO ALLAJO META
# 2. Define the exact target directories inside /kaggle/working
working_dir = "/kaggle/working"
sub_output_dir = os.path.join(working_dir, "grpo_length_reward")

# --- Step A: Create the subfolders inside /kaggle/working ---
os.makedirs(sub_output_dir, exist_ok=True)
print("📁 Target subdirectories verified inside /kaggle/working")

# --- Step B: Copy Checkpoint into the root of /kaggle/working ---
src_checkpoint = os.path.join(input_dataset_dir, "grpo_checkpoint_length_1024v2.pkl")
dst_checkpoint = os.path.join(working_dir, "grpo_checkpoint_length.pkl")

if os.path.exists(src_checkpoint):
    shutil.copy(src_checkpoint, dst_checkpoint)
    print("✅ Checkpoint perfectly placed at: /kaggle/working/grpo_checkpoint_length.pkl")
else:
    print(f"❌ Looking for: {src_checkpoint}")
    print("Could not find 'grpo_checkpoint_pure.pkl' in that specific directory. Check file spelling inside the dataset.")

# --- Step C: Copy length log into the subfolder ---
# Checking both 'length_log.csv' and 'length_log' just in case the extension changed during upload
src_log_csv = os.path.join(input_dataset_dir, "length_log (17).csv")
src_log_raw = os.path.join(input_dataset_dir, "length_log")
dst_log = os.path.join(sub_output_dir, "length_log.csv")

if os.path.exists(src_log_csv):
    shutil.copy(src_log_csv, dst_log)
    print("✅ Metrics log perfectly placed inside subfolder at: /kaggle/working/grpo_pure_outcome/length_log.csv")
elif os.path.exists(src_log_raw):
    shutil.copy(src_log_raw, dst_log)
    print("✅ Metrics log (renamed from raw) perfectly placed inside subfolder at: /kaggle/working/grpo_pure_outcome/length_log.csv")
else:
    print(f"❌ Could not find 'length_log.csv' or 'length_log' at: {input_dataset_dir}")

📁 Target subdirectories verified inside /kaggle/working
✅ Checkpoint perfectly placed at: /kaggle/working/grpo_checkpoint_length.pkl
✅ Metrics log perfectly placed inside subfolder at: /kaggle/working/grpo_pure_outcome/length_log.csv


In [4]:
# ============================================================
# CELL 3: Load Llama 3.2 1B Model with HF Token (Kaggle Secrets)
# ============================================================

import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient

# 1. Retrieve HF token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# 2. Set cache directories (prevents filling up disk)
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/hf_cache'
os.makedirs('/kaggle/working/hf_cache', exist_ok=True)

# 3. Model name (gated Llama 1B)
model_name = "meta-llama/Llama-3.2-1B-Instruct"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)
print("✅ Model loaded successfully!")

# 4. Quick inference test
prompt = "What is 17 + 25? Let's think step by step."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- Model Response ---")
print(response)

Loading meta-llama/Llama-3.2-1B-Instruct...


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


✅ Model loaded successfully!

--- Model Response ---
What is 17 + 25? Let's think step by step. 

First, we add the numbers together: 
17 + 25 = 42

So, the answer is 42.


In [8]:
import os
import json

# === 1. Verify training data ===
data_path = '/kaggle/working/Emergence-of-Thinking/data/math_formatted.jsonl'
print('Training data exists:', os.path.exists(data_path))

if os.path.exists(data_path):
    with open(data_path, 'r') as f:
        sample = json.loads(f.readline())
    print('Sample keys:', list(sample.keys()))
    print('Sample input (first 200 chars):', sample.get('input', '')[:200])
else:
    print('WARNING: Training data not found. Check the repo structure.')

# === 2. Verify evaluation data ===
eval_data = '/kaggle/working/Emergence-of-Thinking/evaluation/data/math'
print('Eval data exists:', os.path.exists(eval_data))

# === 3. Create necessary directories (Kaggle paths) ===
os.makedirs('/kaggle/working/tmp_code', exist_ok=True)
os.makedirs('/kaggle/working/Emergence-of-Thinking/logs/openrlhf_train_ppo', exist_ok=True)
print('Directories ready.')

Training data exists: True
Sample keys: ['input', 'problem', 'solution', 'answer']
Sample input (first 200 chars): <|start_header_id|>user<|end_header_id|>

Problem: Ryan has 3 red lava lamps and 3 blue lava lamps. He arranges them in a row on a shelf randomly, then turns 3 random lamps on. What is the probability
Eval data exists: True
Directories ready.


In [7]:
import shutil, os

# Remove previous checkpoint directory if it exists (clean start)
checkpoint_dir = '/kaggle/working/checkpoints'
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir, ignore_errors=True)
    print(f'Removed {checkpoint_dir}')
else:
    print(f'No existing {checkpoint_dir} to remove.')

# Remove Ray temporary directory (common source of leftover files)
ray_tmp = '/tmp/ray'
if os.path.exists(ray_tmp):
    shutil.rmtree(ray_tmp, ignore_errors=True)
    print(f'Removed {ray_tmp}')
else:
    print(f'No existing {ray_tmp} to remove.')

# Check disk usage for Kaggle's working directory (main storage)
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'/kaggle/working -- Free: {free/1e9:.1f} GB')

# Check /tmp (temporary storage, often memory‑backed)
total, used, free = shutil.disk_usage('/tmp')
print(f'/tmp -- Free: {free/1e9:.1f} GB')

print('Ready for training')

No existing /kaggle/working/checkpoints to remove.
No existing /tmp/ray to remove.
/kaggle/working -- Free: 20.9 GB
/tmp -- Free: 1180.0 GB
Ready for training


In [8]:
import os

# Path to deepspeed.py inside the cloned repo (Kaggle path)
filepath = '/kaggle/working/Emergence-of-Thinking/openrlhf/utils/deepspeed/deepspeed.py'

if os.path.exists(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()

    patched = False
    new_lines = []
    for line in lines:
        if 'assert state_dict_keys.issubset(' in line and not patched:
            indent = len(line) - len(line.lstrip())
            new_lines.append(' ' * indent + 'state_dict_keys -= {"base_model.model.lm_head.weight"}\n')
            patched = True
        new_lines.append(line)

    if patched:
        with open(filepath, 'w') as f:
            f.writelines(new_lines)
        print('✅ DeepSpeed patched successfully')
    else:
        print('ℹ️ Already patched or pattern not found')
else:
    print(f'❌ File not found: {filepath}')

✅ DeepSpeed patched successfully


### EVAL CONFIG

installation of latex2sympy2

In [9]:
import subprocess, sys

# Install latex2sympy2 (needed for answer equivalence)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'latex2sympy2', '-q'], check=True)
print('latex2sympy2 installed')

# Test import
from latex2sympy2 import latex2sympy
print('Import OK')

# Do NOT start the ORM server – we'll use a local reward function instead
print('ORM server skipped – using local reward function in GRPO.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.7.2 which is incompatible.


latex2sympy2 installed
Import OK
ORM server skipped – using local reward function in GRPO.


### GRPO

In [10]:
!pip install peft datasets latex2sympy2 -q

In [30]:
%%writefile /kaggle/working/Emergence-of-Thinking/train_grpo_resumable.py
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import json
import numpy as np
import random
import os
import pickle
import argparse
from tqdm import tqdm
import re
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Minimize memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Try loading latex2sympy safely
try:
    from latex2sympy2 import latex2sympy
except ImportError:
    latex2sympy = None

# ---------- Reward function ----------
def extract_answer(text):
    match = re.search(r'\\boxed{([^}]+)}', text)
    if match:
        return match.group(1).strip()
    numbers = re.findall(r'-?\d+\.?\d*', text)
    # if numbers:
    #     return numbers[-1]
    return None

def compute_reward(response, ground_truth, tokenizer, alpha=0.8, C=1.0):
    correct = 0.0
    pred_answer = extract_answer(response)
    if pred_answer is None:
        correct = -0.5
    elif pred_answer is not None and latex2sympy is not None:
        try:
            pred_expr = latex2sympy(pred_answer)
            true_expr = latex2sympy(ground_truth)
            if pred_expr == true_expr:
                correct = 1.0
            else:
                try:
                    pred_float = float(pred_expr)
                    true_float = float(true_expr)
                    if abs(pred_float - true_float) < 1e-6:
                        correct = 1.0
                except (TypeError, ValueError):
                    pass
        except Exception:
            if str(pred_answer).strip() == str(ground_truth).strip():
                correct = 1.0
    else:
        if str(pred_answer).strip() == str(ground_truth).strip():
            correct = 1.0
                
    tokens = tokenizer.encode(response, add_special_tokens=False)
    num_tokens = max(len(tokens), 1)
    exploration_reward = -C / num_tokens

    reward = alpha * correct + (1 - alpha) * exploration_reward
    reward = min(10, max(-10,reward))
    return reward, correct

# ---------- GRPO training ----------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="meta-llama/Llama-3.2-1B-Instruct")
    parser.add_argument("--data_path", type=str, default="/kaggle/working/Emergence-of-Thinking/data/math_formatted.jsonl")
    parser.add_argument("--output_dir", type=str, default="/kaggle/working/grpo_checkpoints")
    parser.add_argument("--resume_from", type=str, default="/kaggle/working/grpo_checkpoint.pkl")
    parser.add_argument("--max_samples", type=int, default=1000)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--group_size", type=int, default=3)
    parser.add_argument("--max_new_tokens", type=int, default=256)
    parser.add_argument("--learning_rate", type=float, default=1e-5)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--checkpoint_every", type=int, default=20)
    parser.add_argument("--kl_coef", type=float, default=0.01)
    parser.add_argument("--reward_alpha", type=float, default=0.8)
    parser.add_argument("--reward_C", type=float, default=1.0)
    parser.add_argument("--use_lora", action="store_true", default=True)
    parser.add_argument("--lora_r", type=int, default=16)
    parser.add_argument("--lora_alpha", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=8) # Added Accumulation Steps
    args = parser.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    random.seed(args.seed)

    os.makedirs(args.output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    # ========= Χρήση auto device map & SDPA =========
    print("Loading Policy Model to GPU 0...")
    policy_model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        quantization_config=bnb_config,
        device_map={"":"cuda:0"},
        attn_implementation="sdpa"
    )
    print("=====================")
    print(f"Policy Model layer 0 dtype: {policy_model.model.layers[0].self_attn.q_proj.weight.dtype}")
    print("=====================")
    # ========= 4-BIT GRADIENT CHECKPOINTING =========
    policy_model = prepare_model_for_kbit_training(policy_model)
    if args.use_lora:
        lora_config = LoraConfig(
            r=args.lora_r,
            lora_alpha=args.lora_alpha,
            target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        policy_model = get_peft_model(policy_model, lora_config)
        policy_model.print_trainable_parameters()

    print("Loading Reference Model to GPU 1...")
    ref_model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        quantization_config=bnb_config,
        device_map={"":"cuda:1"},
        attn_implementation="sdpa"
    )
    ref_model.eval()
    for param in ref_model.parameters():
        param.requires_grad = False

    examples = []
    with open(args.data_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= args.max_samples:
                break
            data = json.loads(line)
            examples.append({"prompt": data["input"], "answer": data["answer"]})
    dataset = Dataset.from_list(examples)

    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=args.learning_rate)
    
    # Calculate effective steps considering accumulation
    steps_per_epoch = len(dataset) // (args.batch_size * args.gradient_accumulation_steps)
    total_steps = args.epochs * max(1, steps_per_epoch) # Avoid 0 if dataset is too small
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    start_step = 0
    start_epoch = 0
    
    if os.path.exists(args.resume_from):
        print(f"Resuming from {args.resume_from}")
        with open(args.resume_from, 'rb') as f:
            checkpoint = pickle.load(f)
        
        policy_model.load_state_dict(checkpoint['policy_state_dict'], strict=False)
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_step = checkpoint['step']
        start_epoch = checkpoint.get('epoch', 0)
            
        if 'torch_rng' in checkpoint:
            torch.set_rng_state(checkpoint['torch_rng'])
            np.random.set_state(checkpoint['numpy_rng'])
            random.setstate(checkpoint['random_rng'])
            
        print(f"Resumed successfully. Clean restart from global step {start_step + 1} on Epoch {start_epoch}...")
    else:
        print("Starting fresh training")

    policy_model.train()
    optimizer.zero_grad() # Initialize clear gradients

    log_path = os.path.join(args.output_dir, 'length_log.csv')
    if not os.path.exists(log_path):
        with open(log_path, 'w') as f:
            f.write("step,avg_token_length,kl_divergence,accuracy\n")

    for epoch in range(start_epoch, args.epochs):
        indices = list(range(len(dataset)))
        random.seed(args.seed + epoch)
        random.shuffle(indices)
        
        batches = [indices[i:i + args.batch_size] for i in range(0, len(indices), args.batch_size)]
        batches = [b for b in batches if len(b) == args.batch_size]
        
        if epoch == start_epoch:
            completed_batches_in_epoch = start_step % len(batches)
        else:
            completed_batches_in_epoch = 0
            
        active_batches = batches[completed_batches_in_epoch:]
        
        pbar = tqdm(
            active_batches,
            desc=f"Epoch {epoch}",
            initial=completed_batches_in_epoch,
            total=len(batches)
        )

        # Track total expected updates for logging
        total_updates_per_epoch = (len(batches) + args.gradient_accumulation_steps - 1) // args.gradient_accumulation_steps
        total_expected_updates = args.epochs * total_updates_per_epoch
        global_update_step = start_step // args.gradient_accumulation_steps

        # Initialize Cycle Trackers
        cycle_rewards = []
        cycle_correctness = []
        cycle_lengths = []
        cycle_kl_sum = 0.0
        cycle_sequences_count = 0
        accumulated_display_loss = 0.0
        
        for batch_idx, batch_indices in enumerate(pbar):
            step = (epoch * len(batches)) + completed_batches_in_epoch + batch_idx + 1

            prompts = [dataset[i]["prompt"] for i in batch_indices]
            answers = [dataset[i]["answer"] for i in batch_indices]

            all_responses = []
            all_rewards = []
            all_correctness = []
            all_full_tokens = []
            all_prompt_lens = []

            for prompt, ans in zip(prompts, answers):
                # Using policy_model.device usually points to cuda:0 which is perfect for inputs
                inputs = tokenizer(prompt, return_tensors="pt").to(policy_model.device)
                input_ids = inputs.input_ids
                attention_mask = inputs.attention_mask
                prompt_len = input_ids.shape[1]
                
                batched_input_ids = input_ids.repeat(args.group_size, 1)
                batched_attention_mask = attention_mask.repeat(args.group_size, 1)
                
                # =====================================================
                policy_model.eval() # disable dropout of LoRA
                policy_model.gradient_checkpointing_disable() # Permit Caching
                with torch.no_grad():
                    output_ids = policy_model.generate(
                        batched_input_ids,
                        attention_mask=batched_attention_mask,
                        max_new_tokens=args.max_new_tokens,
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.95,
                        repetition_penalty=1.15,
                        pad_token_id=tokenizer.eos_token_id,
                        use_cache=True
                    )
                policy_model.train() # Back to train mode 
                policy_model.gradient_checkpointing_enable() # Enable again gradient checkpoint to save VRAM during backward pass
                # =====================================================
                group_token_lengths = [] # for tracking
                for sample_idx in range(args.group_size):
                    single_output = output_ids[sample_idx]
                    response_ids = single_output[prompt_len:]
                    response = tokenizer.decode(response_ids, skip_special_tokens=True)
                    
                    reward, is_correct = compute_reward(response, ans, tokenizer, alpha=args.reward_alpha, C=args.reward_C)

                    token_count = len(tokenizer.encode(response, add_special_tokens=False))
                    group_token_lengths.append(token_count)
                    all_responses.append(response)
                    all_rewards.append(reward)
                    all_correctness.append(is_correct)
                    
                    full_seq = torch.cat([input_ids[0], response_ids], dim=-1)
                    all_full_tokens.append(full_seq)
                    all_prompt_lens.append(prompt_len)

            # Add this micro-batch's stats to the cycle pile
            cycle_lengths.extend(group_token_lengths)
            cycle_rewards.extend(all_rewards)
            cycle_correctness.extend(all_correctness)
            
            tqdm.write(f"  > Batch {batch_idx}, Batch Indices {batch_indices} Sample Group Lengths: {group_token_lengths}")
            rewards_tensor = torch.tensor(all_rewards, dtype=torch.float32, device=policy_model.device)
            
            advantages = []
            for i in range(len(prompts)):
                start = i * args.group_size
                end = (i+1) * args.group_size
                group_rewards = rewards_tensor[start:end]
                mean_r = group_rewards.mean()
                std_r = group_rewards.std() + 1e-8
                group_adv = (group_rewards - mean_r) / std_r
                advantages.append(group_adv)
            advantages = torch.cat(advantages)

            del all_responses
            del all_rewards
            torch.cuda.empty_cache()

            loss_fct = torch.nn.CrossEntropyLoss(reduction='none')

            total_sequences = len(all_full_tokens)
            step_kl_sum = 0.0
            
            for idx in range(total_sequences):
                prompt_len = all_prompt_lens[idx]
                
                sub_batch_p = all_full_tokens[idx].unsqueeze(0).to(policy_model.device)
                sub_batch_r = all_full_tokens[idx].unsqueeze(0).to(ref_model.device)

                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    # Forward Policy
                    outputs_p = policy_model(sub_batch_p)
                    logits_p = outputs_p.logits[0, prompt_len-1:-1, :].contiguous()
                    
                    seq_labels_p = sub_batch_p[0, prompt_len:].contiguous()
                    non_pad_mask_p = (seq_labels_p != tokenizer.eos_token_id)
    
                    seq_len = torch.clamp(non_pad_mask_p.sum(), min=1.0)
                    
                    loss_p = loss_fct(logits_p, seq_labels_p)
                    log_prob_p = (-loss_p * non_pad_mask_p).sum()
                    
                    # Forward Reference
                    with torch.no_grad():
                        outputs_r = ref_model(sub_batch_r)
                        logits_r = outputs_r.logits[0, prompt_len-1:-1, :].contiguous()
                    
                    # Μεταφορά των reference logits και του adv στην ίδια κάρτα με τα policy logits
                    logits_r = logits_r.to(logits_p.device)
                    adv = advantages[idx].to(logits_p.device)
                    
                    log_probs_p = F.log_softmax(logits_p, dim=-1)
                    log_probs_r = F.log_softmax(logits_r, dim=-1)
                    
                    kl_token_wise = (log_probs_p.exp() * (log_probs_p - log_probs_r)).sum(dim=-1)
                    kl_div = (kl_token_wise * non_pad_mask_p).sum()

                    # Accumulate the raw KL values for stats
                    step_kl_sum += kl_div.sum().item()
                    
                    seq_loss = -(adv * log_prob_p - args.kl_coef * kl_div) / total_sequences
                    seq_loss_ga = seq_loss / args.gradient_accumulation_steps
                
                seq_loss_ga.backward()
                accumulated_display_loss += seq_loss.item() # Keep display math aligned with unscaled loss
                
                del outputs_p, logits_p, outputs_r, logits_r, loss_p
                if idx % 2 == 0:
                    torch.cuda.empty_cache()

            # Add sequence count and KL to the cycle pile
            cycle_kl_sum += step_kl_sum
            cycle_sequences_count += total_sequences

            # Update weights only when enough gradients have accumulated
            if (batch_idx + 1) % args.gradient_accumulation_steps == 0 or (batch_idx + 1) == len(batches):
                torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad() # Clear for the next accumulation cycle
                
                global_update_step += 1

                # Calculate the true averages for the entire accumulated batch
                true_mean_reward = sum(cycle_rewards) / len(cycle_rewards) if cycle_rewards else 0.0
                true_accuracy = sum(cycle_correctness) / len(cycle_correctness) if cycle_correctness else 0.0
                true_avg_len = sum(cycle_lengths) / len(cycle_lengths) if cycle_lengths else 0.0
                true_avg_kl = cycle_kl_sum / max(cycle_sequences_count, 1)

                # Write everything to the CSV using the global step
                with open(log_path, 'a') as f:
                    f.write(f"{global_update_step},{true_avg_len:.2f},{true_avg_kl:.4f},{true_accuracy:.4f}\n")

                tqdm.write(f"UPDATE {global_update_step}/{total_expected_updates} (Epoch {epoch}) | Loss: {accumulated_display_loss:.4f} | Reward: {true_mean_reward:.4f} | Acc: {true_accuracy:.4f} | Len: {true_avg_len:.1f} | KL: {true_avg_kl:.4f}")

                if global_update_step % args.checkpoint_every == 0:
                    checkpoint = {
                        'policy_state_dict': policy_model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(),
                        'step': step, # Kept micro-step for accurate dataset resuming
                        'epoch': epoch,
                        'torch_rng': torch.get_rng_state(),
                        'numpy_rng': np.random.get_state(),
                        'random_rng': random.getstate(),
                    }
                    with open(args.resume_from, 'wb') as f:
                        pickle.dump(checkpoint, f)
                    
                    policy_model.save_pretrained(args.output_dir)
                    tokenizer.save_pretrained(args.output_dir)
                    tqdm.write(f"Checkpoint saved and model adapters updated at UPDATE {global_update_step}")

                # Reset trackers for the next cycle
                cycle_rewards = []
                cycle_correctness = []
                cycle_lengths = []
                cycle_kl_sum = 0.0
                cycle_sequences_count = 0
                accumulated_display_loss = 0.0

    policy_model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print(f"Training complete. Final model saved to {args.output_dir}")

if __name__ == "__main__":
    main()

Overwriting /kaggle/working/Emergence-of-Thinking/train_grpo_resumable.py


**++++EXPLORATION**

In [33]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ['HF_TOKEN'] = hf_token
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/hf_cache'

!python /kaggle/working/Emergence-of-Thinking/train_grpo_resumable.py \
    --model_name meta-llama/Llama-3.2-1B-Instruct \
    --data_path /kaggle/working/Emergence-of-Thinking/data/math_formatted.jsonl \
    --max_samples 5500 \
    --batch_size 2 \
    --group_size 3 \
    --gradient_accumulation_steps 8 \
    --max_new_tokens 2200 \
    --learning_rate 1e-5 \
    --epochs 5 \
    --checkpoint_every 2 \
    --kl_coef 0.05 \
    --reward_alpha 0.8 \
    --reward_C 1000.0 \
    --output_dir /kaggle/working/grpo_length_reward_gradientaccumulationplusnoboxpenalty \
    --resume_from /kaggle/working/grpo_checkpoint_length_gradientaccumulationplusnoboxpenalty.pkl

/root/.local/lib/python3.12/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-06-22 02:01:42.923577: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782093702.947013     446 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782093702.955651     446 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782093702.975404     446 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782093702.975435     446 computation_

### Evaluation

In [29]:
!rm -rf /kaggle/working/grpo_length_reward_gradientaccumulationplusnoboxpenalty

In [ ]:
%%writefile /kaggle/working/Emergence-of-Thinking/evaluate_grpo_math.py
import torch
import json
import re
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm import tqdm
from latex2sympy2 import latex2sympy
import argparse

def extract_answer(text):
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    if match:
        return match.group(1).strip()
    numbers = re.findall(r'-?\d+\.?\d*', text)
    if numbers:
        return numbers[-1]
    return None

def is_correct(response, ground_truth):
    pred = extract_answer(response)
    if pred is None:
        return False
    try:
        pred_expr = latex2sympy(pred)
        true_expr = latex2sympy(ground_truth)
        if pred_expr == true_expr:
            return True
        try:
            pred_float = float(pred_expr)
            true_float = float(true_expr)
            return abs(pred_float - true_float) < 1e-6
        except (TypeError, ValueError):
            pass
    except Exception:
        pass
    return str(pred).strip() == str(ground_truth).strip()

def format_prompt(problem):
    return f"<|start_header_id|>user<|end_header_id|>\n\nProblem: {problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_path", type=str, required=True)
    parser.add_argument("--base_model", type=str, default="meta-llama/Llama-3.2-1B-Instruct")
    parser.add_argument("--eval_file", type=str, default="/kaggle/working/Emergence-of-Thinking/evaluation/data/math/test.jsonl")
    parser.add_argument("--max_new_tokens", type=int, default=1024)
    args = parser.parse_args()

    tokenizer = AutoTokenizer.from_pretrained(args.base_model)
    tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    model = PeftModel.from_pretrained(base, args.model_path)
    model.eval()

    problems = []
    with open(args.eval_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            problems.append((data['problem'], data['answer']))
    print(f"Loaded {len(problems)} test problems from {args.eval_file}")

    correct = 0
    for problem, answer in tqdm(problems):
        prompt = format_prompt(problem)
        # Fix: Prevent duplicate special token prefix injections
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=args.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        if is_correct(response, answer):
            correct += 1
            
    acc = correct / len(problems) if len(problems) > 0 else 0
    print(f"\nTarget Path: {args.model_path}")
    print(f"Accuracy: {acc:.4f} ({correct}/{len(problems)})")

if __name__ == "__main__":
    main()

In [ ]:
# Evaluate Run A: Pure Outcome
!python /kaggle/working/Emergence-of-Thinking/evaluate_grpo_math.py \
    --model_path /kaggle/working/grpo_pure_outcome \
    --eval_file /kaggle/working/Emergence-of-Thinking/evaluation/data/math/test.jsonl \
    --max_new_tokens 1024

# Evaluate Run B: Length Reward
!python /kaggle/working/Emergence-of-Thinking/evaluate_grpo_math.py \
    --model_path /kaggle/working/grpo_length_reward \
    --eval_file /kaggle/working/Emergence-of-Thinking/evaluation/data/math/test.jsonl \
    --max_new_tokens 1024

### Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

pure_log_path = '/kaggle/working/grpo_pure_outcome/length_log.csv'
length_log_path = '/kaggle/working/grpo_length_reward/length_log.csv'

plt.figure(figsize=(10, 5))

if os.path.exists(pure_log_path):
    df_pure = pd.read_csv(pure_log_path)
    plt.plot(df_pure['step'], df_pure['avg_token_length'], label='Pure outcome (α=1.0)', color='crimson')

if os.path.exists(length_log_path):
    df_len = pd.read_csv(length_log_path)
    plt.plot(df_len['step'], df_len['avg_token_length'], label='Outcome + length (α=0.8, C=1.0)', color='teal')

plt.xlabel('Training step')
plt.ylabel('Average response length (tokens)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.title('Length Evolution during GRPO on Llama 3.2 1B')
plt.show()

### Second Result (SFT + RLSP)

In [13]:
%%writefile /kaggle/working/Emergence-of-Thinking/train_sft_local.py
import torch
import json
import argparse
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ['HF_TOKEN'] = hf_token

def clean_text(text):
    """Αφαίρεση κενών, σημείων στίξης και μετατροπή σε πεζά για ακριβή σύγκριση."""
    if not text:
        return ""
    return re.sub(r'\W+', '', text.lower().strip())

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="meta-llama/Llama-3.2-1B-Instruct")
    # Χρήση του επίσημου Hugging Face dataset ID
    parser.add_argument("--dataset_name", type=str, default="Logics-MLLM/Logics-STEM-SFT-Dataset-Open-1.6M")
    parser.add_argument("--output_dir", type=str, default="/kaggle/working/sft_math_local")
    parser.add_argument("--max_samples", type=int, default=2000)
    parser.add_argument("--batch_size", type=int, default=2)
    parser.add_argument("--grad_accum", type=int, default=4)
    parser.add_argument("--epochs", type=int, default=3)   
    parser.add_argument("--lr", type=float, default=2e-5)
    args = parser.parse_args()

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    tokenizer.pad_token = tokenizer.eos_token

    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.bfloat16,
        device_map={"": local_rank} 
    )
    
    # LoRA Config
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)

    # 1. Φόρτωση του MATH500 για τη δημιουργία της λίστας αποκλεισμού
    print("Φόρτωση του MATH500 για έλεγχο data leakage...")
    math500 = load_dataset("HuggingFaceH4/MATH-500", split="test", token=hf_token)
    math500_problems = {clean_text(ex["problem"]) for ex in math500 if ex.get("problem")}
    print(f"Βρέθηκαν {len(math500_problems)} μοναδικά προβλήματα προς φιλτράρισμα.")

    # 2. Streaming του τεράστιου dataset εκπαίδευσης
    print(f"Έναρξη streaming του dataset: {args.dataset_name}...")
    raw_stream = load_dataset(args.dataset_name, split="train", streaming=True, token=hf_token)

    examples = []
    leakage_count = 0

    for item in raw_stream:
        if len(examples) >= args.max_samples:
            break
            
        # Εξαγωγή κειμένου ανάλογα με τη δομή (μηνύματα ή instruction/output)
        if "messages" in item:
            user_prompt = next((m["content"] for m in item["messages"] if m["role"] == "user"), "")
            assistant_response = next((m["content"] for m in item["messages"] if m["role"] == "assistant"), "")
        else:
            user_prompt = item.get("query", item.get("instruction", item.get("input", "")))
            assistant_response = item.get("response", item.get("output", item.get("solution", "")))

        # Έλεγχος για Data Leakage
        if clean_text(user_prompt) in math500_problems:
            leakage_count += 1
            continue  # Παράλειψη αυτού του δείγματος

        # Μορφοποίηση για Causal LM SFT
        full_text = f"Question: {user_prompt}\nReasoning and Solution:\n{assistant_response}{tokenizer.eos_token}"
        examples.append({"text": full_text})

    print(f"Συλλέχθηκαν {len(examples)} καθαρά δείγματα.")
    print(f"Απορρίφθηκαν {leakage_count} προβλήματα λόγω διαρροής (leakage).")

    # Μετατροπή της λίστας σε Hugging Face Dataset object
    dataset = Dataset.from_list(examples)

    def tokenize(examples):
        # Αυξάνουμε ελαφρώς το max_length (π.χ. 1024) επειδή τα CoT δεδομένα είναι μεγάλα
        return tokenizer(examples["text"], truncation=True, max_length=1024)
        
    tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    training_args = TrainingArguments(
        output_dir=args.output_dir,
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        num_train_epochs=args.epochs,
        logging_steps=10,
        save_steps=250,
        bf16=True,
        report_to="none"
    )
    
    trainer = Trainer(model=model, args=training_args, train_dataset=tokenized, data_collator=data_collator)
    trainer.train()
    
    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

if __name__ == "__main__":
    main()


Writing /kaggle/working/Emergence-of-Thinking/train_sft_local.py


In [14]:
# !cd /kaggle/working/Emergence-of-Thinking && python train_sft_local.py \
!cd /kaggle/working/Emergence-of-Thinking && accelerate launch --multi_gpu --num_processes 2 train_sft_local.py \
    --model_name meta-llama/Llama-3.2-1B-Instruct \
    --dataset_name Logics-MLLM/Logics-STEM-SFT-Dataset-Open-1.6M \
    --max_samples 2000 \
    --batch_size 2 \
    --grad_accum 4 \
    --epochs 2 \
    --output_dir /kaggle/working/sft_math_local

/root/.local/lib/python3.12/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/root/.local/lib/python3.12/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/root/.local/lib/python3.12/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-06-17 16:40:43.545434

In [31]:
import shutil

# Move the weights to the output directory
shutil.copytree('/kaggle/working/sft_math_local/', '/kaggle/output/sft_math_weights/', dirs_exist_ok=True)
print("✅ Weights moved to Output directory.")

✅ Weights moved to Output directory.


In [44]:
%%writefile /kaggle/working/Emergence-of-Thinking/train_grpo_from_sft.py
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import Dataset
import json
import numpy as np
import random
import os
import pickle
import argparse
from tqdm import tqdm
import re
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ['HF_TOKEN'] = hf_token
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/hf_cache'

# Minimize memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Try loading latex2sympy safely
try:
    from latex2sympy2 import latex2sympy
except ImportError:
    latex2sympy = None

# ---------- Reward function ----------
def extract_answer(text):
    match = re.search(r'\\boxed{([^}]+)}', text)
    if match:
        return match.group(1).strip()
    numbers = re.findall(r'-?\d+\.?\d*', text)
    if numbers:
        return numbers[-1]
    return None

def compute_reward(response, ground_truth, tokenizer, alpha=0.8, C=1.0):
    correct = 0.0
    pred_answer = extract_answer(response)
    if pred_answer is not None and latex2sympy is not None:
        try:
            pred_expr = latex2sympy(pred_answer)
            true_expr = latex2sympy(ground_truth)
            if pred_expr == true_expr:
                correct = 1.0
            else:
                try:
                    pred_float = float(pred_expr)
                    true_float = float(true_expr)
                    if abs(pred_float - true_float) < 1e-6:
                        correct = 1.0
                except (TypeError, ValueError):
                    pass
        except Exception:
            if str(pred_answer).strip() == str(ground_truth).strip():
                correct = 1.0
    else:
        if str(pred_answer).strip() == str(ground_truth).strip():
            correct = 1.0
                
    tokens = tokenizer.encode(response, add_special_tokens=False)
    num_tokens = max(len(tokens), 1)
    exploration_reward = -C / num_tokens

    reward = alpha * correct + (1 - alpha) * exploration_reward
    reward = min(10, max(-10,reward))
    return reward

# ---------- GRPO training ----------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="meta-llama/Llama-3.2-1B-Instruct")
    parser.add_argument("--sft_adapter_path", type=str, required=True, help="Path to your trained SFT adapter")
    parser.add_argument("--data_path", type=str, default="/kaggle/working/Emergence-of-Thinking/data/math_formatted.jsonl")
    parser.add_argument("--output_dir", type=str, default="/kaggle/working/grpo_checkpoints")
    parser.add_argument("--resume_from", type=str, default="/kaggle/working/grpo_checkpoint.pkl")
    parser.add_argument("--max_samples", type=int, default=2000)
    parser.add_argument("--batch_size", type=int, default=2)
    parser.add_argument("--group_size", type=int, default=4)
    parser.add_argument("--max_new_tokens", type=int, default=256)
    parser.add_argument("--learning_rate", type=float, default=1e-5)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--checkpoint_every", type=int, default=50)
    parser.add_argument("--kl_coef", type=float, default=0.01)
    parser.add_argument("--reward_alpha", type=float, default=0.8)
    parser.add_argument("--reward_C", type=float, default=1.0)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    random.seed(args.seed)

    os.makedirs(args.output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    print(f"Loading Policy Model (SFT Adapter) to GPU 0...")
    base_policy = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.bfloat16,
        device_map={"":"cuda:0"}
    )
    # Load SFT adapter and explicitly make it trainable for RL
    policy_model = PeftModel.from_pretrained(base_policy, args.sft_adapter_path, is_trainable=True)
    policy_model.print_trainable_parameters()

    print(f"Loading Reference Model (SFT Adapter) to GPU 1...")
    base_ref = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.bfloat16,
        device_map={"":"cuda:1"}
    )
    # Load SFT adapter but keep it FROZEN for the KL penalty baseline
    ref_model = PeftModel.from_pretrained(base_ref, args.sft_adapter_path, is_trainable=False)
    ref_model.eval()
    for param in ref_model.parameters():
        param.requires_grad = False

    examples = []
    with open(args.data_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= args.max_samples:
                break
            data = json.loads(line)
            examples.append({"prompt": data["input"], "answer": data["answer"]})
    dataset = Dataset.from_list(examples)

    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=args.learning_rate)
    steps_per_epoch = len(dataset) // args.batch_size
    total_steps = args.epochs * steps_per_epoch
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    start_step = 0
    start_epoch = 0
    
    if os.path.exists(args.resume_from):
        print(f"Resuming from {args.resume_from}")
        with open(args.resume_from, 'rb') as f:
            checkpoint = pickle.load(f)
        
        policy_model.load_state_dict(checkpoint['policy_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_step = checkpoint['step']
        start_epoch = checkpoint.get('epoch', 0)
        
        if 'torch_rng' in checkpoint:
            torch.set_rng_state(checkpoint['torch_rng'])
            np.random.set_state(checkpoint['numpy_rng'])
            random.setstate(checkpoint['random_rng'])
        print(f"Resumed successfully at step {start_step + 1}...")

    policy_model.train()

    log_path = os.path.join(args.output_dir, 'length_log.csv')
    if not os.path.exists(log_path):
        with open(log_path, 'w') as f:
            f.write("step,avg_token_length\n")

    for epoch in range(start_epoch, args.epochs):
        indices = list(range(len(dataset)))
        random.seed(args.seed + epoch)
        random.shuffle(indices)
        
        batches = [indices[i:i + args.batch_size] for i in range(0, len(indices), args.batch_size)]
        batches = [b for b in batches if len(b) == args.batch_size]
        
        completed_batches_in_epoch = start_step % len(batches) if epoch == start_epoch else 0
        active_batches = batches[completed_batches_in_epoch:]
        
        pbar = tqdm(active_batches, desc=f"Epoch {epoch}", initial=completed_batches_in_epoch, total=len(batches))
        
        for batch_idx, batch_indices in enumerate(pbar):
            step = (epoch * len(batches)) + completed_batches_in_epoch + batch_idx + 1

            prompts = [dataset[i]["prompt"] for i in batch_indices]
            answers = [dataset[i]["answer"] for i in batch_indices]

            all_responses = []
            all_rewards = []
            all_full_tokens = []
            all_prompt_lens = []

            # --- OPTIMIZED BATCH GENERATION ---
            for prompt, ans in zip(prompts, answers):
                inputs = tokenizer(prompt, return_tensors="pt").to(policy_model.device)
                input_ids = inputs.input_ids
                attention_mask = inputs.attention_mask
                prompt_len = input_ids.shape[1]
                
                batched_input_ids = input_ids.repeat(args.group_size, 1)
                batched_attention_mask = attention_mask.repeat(args.group_size, 1)
                
                with torch.no_grad():
                    output_ids = policy_model.generate(
                        batched_input_ids,
                        attention_mask=batched_attention_mask, 
                        max_new_tokens=args.max_new_tokens,
                        do_sample=True,
                        temperature=0.8,
                        pad_token_id=tokenizer.eos_token_id
                    )
                
                for sample_idx in range(args.group_size):
                    single_output = output_ids[sample_idx]
                    response_ids = single_output[prompt_len:]
                    response = tokenizer.decode(response_ids, skip_special_tokens=True)
                    reward = compute_reward(response, ans, tokenizer, alpha=args.reward_alpha, C=args.reward_C)
                    
                    all_responses.append(response)
                    all_rewards.append(reward)
                    
                    full_seq = torch.cat([input_ids[0], response_ids], dim=-1)
                    all_full_tokens.append(full_seq)
                    all_prompt_lens.append(prompt_len)

            avg_token_len = sum(len(tokenizer.encode(r)) for r in all_responses) / len(all_responses)
            with open(log_path, 'a') as f:
                f.write(f"{step},{avg_token_len:.2f}\n")

            rewards_tensor = torch.tensor(all_rewards, dtype=torch.float32, device=policy_model.device)
            
            advantages = []
            for i in range(len(prompts)):
                start = i * args.group_size
                end = (i+1) * args.group_size
                group_rewards = rewards_tensor[start:end]
                mean_r = group_rewards.mean()
                std_r = group_rewards.std() + 1e-8
                group_adv = (group_rewards - mean_r) / std_r
                advantages.append(group_adv)
            advantages = torch.cat(advantages)

            del all_responses
            del all_rewards
            torch.cuda.empty_cache()

            loss_fct = torch.nn.CrossEntropyLoss(reduction='none')

            # --- ULTRA VRAM-SAFE SEQUENTIAL GRADIENT ACCUMULATION ---
            total_sequences = len(all_full_tokens)
            optimizer.zero_grad()
            accumulated_loss = 0.0
            
            for idx in range(total_sequences):
                prompt_len = all_prompt_lens[idx]
                adv = advantages[idx]
                
                sub_batch_p = all_full_tokens[idx].unsqueeze(0).to(policy_model.device)
                sub_batch_r = all_full_tokens[idx].unsqueeze(0).to(ref_model.device)
                
                outputs_p = policy_model(sub_batch_p)
                logits_p = outputs_p.logits[0, prompt_len-1:-1, :].contiguous()
                seq_labels_p = sub_batch_p[0, prompt_len:].contiguous()
                non_pad_mask_p = (seq_labels_p != tokenizer.eos_token_id)
                
                loss_p = loss_fct(logits_p, seq_labels_p)
                log_prob_p = (-loss_p * non_pad_mask_p).sum()
                
                with torch.no_grad():
                    outputs_r = ref_model(sub_batch_r)
                    logits_r = outputs_r.logits[0, prompt_len-1:-1, :].contiguous()
                
                logits_r = logits_r.to(policy_model.device)
                log_probs_p = F.log_softmax(logits_p, dim=-1)
                log_probs_r = F.log_softmax(logits_r, dim=-1)
                
                kl_token_wise = (log_probs_p.exp() * (log_probs_p - log_probs_r)).sum(dim=-1)
                kl_div = (kl_token_wise * non_pad_mask_p).sum()
                
                seq_loss = -(adv * log_prob_p - args.kl_coef * kl_div) / total_sequences
                
                seq_loss.backward()
                accumulated_loss += seq_loss.item()
                
                del outputs_p, logits_p, outputs_r, logits_r, loss_p
                if idx % 2 == 0:
                    torch.cuda.empty_cache()

            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            tqdm.write(f"Epoch {epoch} | Step {step}: loss={accumulated_loss:.4f}, mean_reward={rewards_tensor.mean():.4f}, avg_len={avg_token_len:.1f}")

            if step % args.checkpoint_every == 0:
                checkpoint = {
                    'policy_state_dict': policy_model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'step': step,
                    'epoch': epoch,
                    'torch_rng': torch.get_rng_state(),
                    'numpy_rng': np.random.get_state(),
                    'random_rng': random.getstate(),
                }
                with open(args.resume_from, 'wb') as f:
                    pickle.dump(checkpoint, f)
                
                policy_model.save_pretrained(args.output_dir)
                tokenizer.save_pretrained(args.output_dir)
                tqdm.write(f"Checkpoint saved at step {step}")

    policy_model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print(f"Training complete. Final model saved to {args.output_dir}")

if __name__ == "__main__":
    main()

Overwriting /kaggle/working/Emergence-of-Thinking/train_grpo_from_sft.py


In [45]:
!cd /kaggle/working/Emergence-of-Thinking && python train_grpo_from_sft.py \
    --model_name meta-llama/Llama-3.2-1B-Instruct \
    --sft_adapter_path /kaggle/working/sft_math_local \
    --data_path /kaggle/working/Emergence-of-Thinking/data/math_formatted.jsonl \
    --max_samples 5500 \
    --batch_size 1 \
    --group_size 4 \
    --max_new_tokens 1024 \
    --learning_rate 4e-6 \
    --epochs 5 \
    --checkpoint_every 15 \
    --kl_coef 0.01 \
    --reward_alpha 0.8 \
    --reward_C 1000.0 \
    --output_dir /kaggle/working/grpo_from_sft \
    --resume_from /kaggle/working/grpo_checkpoint_sft.pkl

/root/.local/lib/python3.12/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-06-16 02:25:27.710077: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781576727.732667    1016 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781576727.740195    1016 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781576727.761303    1016 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781576727.761332    1016 computation_

In [ ]:
# Evaluate SFT model
!cd /kaggle/working/Emergence-of-Thinking && python evaluate_grpo_math.py \
    --model_path /kaggle/working/sft_math_local \
    --eval_file evaluation/data/math/test.jsonl

# Evaluate final GRPO model
!cd /kaggle/working/Emergence-of-Thinking && python evaluate_grpo_math.py \
    --model_path /kaggle/working/grpo_from_sft \
    --eval_file evaluation/data/math/test.jsonl

# Evaluate base model (no LoRA)
!cd /kaggle/working/Emergence-of-Thinking && python evaluate_grpo_math.py \
    --model_path meta-llama/Llama-3.2-1B-Instruct \
    --base_only \
    --eval_file evaluation/data/math/test.jsonl

### Third result matrix

In [46]:
# ============================================================
# COMPLETE PIPELINE FOR TABLE 3 (MATH-500)
# ============================================================

# 1. Write the self-consistency evaluation script
%%writefile /kaggle/working/Emergence-of-Thinking/evaluate_sc_budget.py
import torch
import json
import re
import argparse
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm import tqdm
from collections import Counter
from latex2sympy2 import latex2sympy

def extract_answer(text):
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    if match:
        return match.group(1).strip()
    numbers = re.findall(r'-?\d+\.?\d*', text)
    if numbers:
        return numbers[-1]
    return None

def is_correct(response, ground_truth):
    pred = extract_answer(response)
    if pred is None:
        return False
    try:
        pred_expr = latex2sympy(pred)
        true_expr = latex2sympy(ground_truth)
        if pred_expr == true_expr:
            return True
        pred_float = float(pred_expr)
        true_float = float(true_expr)
        return abs(pred_float - true_float) < 1e-6
    except:
        return pred.strip() == ground_truth.strip()

def format_prompt(problem):
    return f"<|start_header_id|>user<|end_header_id|>\n\nProblem: {problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"

def estimate_avg_length(model, tokenizer, problems, num_samples=50, max_new_tokens=256):
    total_tokens = 0
    count = 0
    model.eval()
    for problem, _ in tqdm(problems[:num_samples], desc="Estimating length"):
        prompt = format_prompt(problem)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        total_tokens += len(tokenizer.encode(response))
        count += 1
    return total_tokens / count

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_path", type=str, default=None)
    parser.add_argument("--base_model", type=str, default="meta-llama/Llama-3.2-1B-Instruct")
    parser.add_argument("--eval_file", type=str, default="/kaggle/working/Emergence-of-Thinking/evaluation/data/math/test.jsonl")
    parser.add_argument("--token_budget", type=int, default=8192)
    parser.add_argument("--max_new_tokens", type=int, default=256)
    parser.add_argument("--temperature", type=float, default=0.7)
    parser.add_argument("--base_only", action="store_true")
    args = parser.parse_args()

    tokenizer = AutoTokenizer.from_pretrained(args.base_model)
    tokenizer.pad_token = tokenizer.eos_token

    if args.base_only:
        model = AutoModelForCausalLM.from_pretrained(args.base_model, torch_dtype=torch.bfloat16, device_map="auto")
    else:
        base = AutoModelForCausalLM.from_pretrained(args.base_model, torch_dtype=torch.bfloat16, device_map="auto")
        model = PeftModel.from_pretrained(base, args.model_path)
    model.eval()

    problems = []
    with open(args.eval_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            problems.append((data['problem'], data['answer']))
    print(f"Loaded {len(problems)} test problems")

    avg_length = estimate_avg_length(model, tokenizer, problems, num_samples=min(50, len(problems)),
                                     max_new_tokens=args.max_new_tokens)
    num_samples = int(args.token_budget // avg_length)
    print(f"Estimated avg length: {avg_length:.1f} tokens → using {num_samples} samples/problem")

    correct_majority = 0
    total_tokens_used = 0
    for problem, answer in tqdm(problems, desc="Self-consistency"):
        prompt = format_prompt(problem)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        responses = []
        for _ in range(num_samples):
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=args.max_new_tokens, do_sample=True,
                                         temperature=args.temperature, pad_token_id=tokenizer.eos_token_id)
            response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            responses.append(response)
            total_tokens_used += len(tokenizer.encode(response))
        answers = [extract_answer(r) for r in responses]
        valid_answers = [a for a in answers if a is not None]
        if valid_answers:
            most_common = Counter(valid_answers).most_common(1)[0][0]
            if most_common == answer or (is_correct(most_common, answer) if most_common else False):
                correct_majority += 1
        else:
            if is_correct(responses[0], answer):
                correct_majority += 1

    accuracy = correct_majority / len(problems)
    avg_tokens = total_tokens_used / len(problems)
    print(f"\nSelf-consistency accuracy: {accuracy:.4f} ({correct_majority}/{len(problems)})")
    print(f"Avg tokens/problem: {avg_tokens:.1f} (budget: {args.token_budget})")

if __name__ == "__main__":
    main()

# 2. Run self-consistency on base model (ONCE)
print("\n=== Running self-consistency evaluation ===\n")
!cd /kaggle/working/Emergence-of-Thinking && python evaluate_sc_budget.py \
    --base_only --token_budget 8192 --eval_file evaluation/data/math/test.jsonl \
    --max_new_tokens 256 --temperature 0.7 > /kaggle/working/sc_output.txt 2>&1

# Capture and parse the output
with open('/kaggle/working/sc_output.txt', 'r') as f:
    sc_output = f.read()
print(sc_output)

import re
sc_acc = float(re.search(r"Self-consistency accuracy:\s+([0-9.]+)", sc_output).group(1))
avg_samples = int(re.search(r"using (\d+) samples per problem", sc_output).group(1))

# 3. Run best-performance evaluation on SFT+GRPO model
print("\n=== Running best-performance evaluation ===\n")
!cd /kaggle/working/Emergence-of-Thinking && python evaluate_grpo_math.py \
    --model_path /kaggle/working/grpo_from_sft \
    --eval_file evaluation/data/math/test.jsonl > /kaggle/working/best_output.txt 2>&1

with open('/kaggle/working/best_output.txt', 'r') as f:
    best_output = f.read()
print(best_output)
best_acc = float(re.search(r"Accuracy:\s+([0-9.]+)", best_output).group(1))

# 4. Print final matrix
print("\n" + "="*50)
print("Final Matrix (Table 3) – Llama-3.2-1B")
print("="*50)
print("| METRIC               | Value                     |")
print("|----------------------|---------------------------|")
print("| BENCHMARK            | MATH-500                  |")
print(f"| AVG SAMPLES          | {avg_samples}                            |")
print(f"| SC ACCURACY          | {sc_acc*100:.1f}%                       |")
print(f"| SFT+ER-RLSP          | {best_acc*100:.1f}%                       |")

UsageError: Line magic function `%%writefile` not found.
